# B2-024 — Practice p19

*65 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: optional-colab-l4` · seed `20261022` · the CPU correctness path below is required and is what CI runs; the `Accelerator extension` section is optional, runs only on a Colab L4, and is never executed by CI  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

**Data boundary:** use literal unit data and the seeded unit generator (`scripts/generate_capstone_data.py`, read through `data/capstone_data.py`), with the standard library, NumPy, and Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints you did not create in the same run, or student data.

**Set:** C · **Type:** integrative · **Difficulty:** advanced · **Time budget:** 65 minutes  
**Concepts:** scientific-ml-inverse-problems

## Task

Train a learned inverse for the sensor-field problem and certify it against the grid-Tikhonov baseline with a physics-residual bound.

**Data.** `x, y = capstone_data.train_rows("inverse")`: 800 float32 rows of 8 noisy sensor magnitudes and targets `(x, y, strength)`, simulated from a source in $[0.1,0.9]^2$ with strength in $[0.5,2]$ (Session 5). Validation: `capstone_data.val_rows("inverse")` (200 rows). The 9×9 grid has 81 unknowns for 8 sensors, so the grid-Tikhonov baseline is heavily underdetermined; it scores a mean test position error of `0.2137` at the frozen seed (`capstone_data.baseline_score("inverse")`). That is the number to beat.

**Model.** `class LearnedInverse(nn.Module)` with constructor `LearnedInverse(train_magnitudes, hidden=64)` that first registers buffers `mu` and `sd` as the mean and standard deviation (`torch.std` default) over rows of `torch.log(train_magnitudes)`, then creates `self.net = nn.Sequential(nn.Linear(8, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 3))`. `forward(m)` takes raw magnitudes `(B, 8)` and returns `self.net((torch.log(m) - self.mu) / self.sd)`.
**Config.** Every device, precision, size, and schedule knob is read from one config dict with the keys of the supplied `CPU_CONFIG`, which is the path CI runs; the accelerator extension changes only the dictionary.
The training pool `(x_fit, y_fit)` is `(x, y)` when `config["simulated_pairs"] == 0` (the CPU path), and otherwise `(x, y)` concatenated with `capstone_data.simulate("inverse", config["simulated_pairs"], seed=3)`.
Construct the model as `torch.manual_seed(SEED)` then `model = LearnedInverse(x_fit, config["hidden"]).to(torch.device(config["device"]))` (on the CPU this is `LearnedInverse(x)` with `hidden=64`).

**Training.** `optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"], betas=(0.9, 0.999), eps=1e-8, weight_decay=0)` (`lr = 3e-3` on the CPU). Write `train_inverse(model, x, y, optimizer, config)` that moves `x, y` to `torch.device(config["device"])` and performs exactly `config["steps"]` (`1500` on the CPU) updates of `F.mse_loss(model(xb), yb)` in train mode, where `(xb, yb)` is the full batch `(x, y)` when `config["batch_size"]` is `None` (the CPU path) and otherwise `(x[idx], y[idx])` for `idx = torch.randint(0, len(x), (config["batch_size"],))` drawn from the global generator. The forward pass and the loss run inside `torch.autocast(device_type=device.type, dtype=config["amp_dtype"])` when `config["amp_dtype"]` is not `None`, with no autocast otherwise. (Each update runs, in this order: `optimizer.zero_grad(set_to_none=True)`, forward, loss, `loss.backward()`, `optimizer.step()`; no scheduler, no gradient clipping, no accumulation, no early stopping.) It returns the list of loss floats (1,500 on the CPU); train with `train_inverse(model, x_fit, y_fit, optimizer, CPU_CONFIG)`.
**Seam:** a forward pre-hook on `model` records `capstone_data.split_of("inverse", row)` for every row of the first training-mode call; the recorded set must be exactly `{"train"}`.

**Physics residual (validation).** Write `forward_field` with the p10 contract. For validation predictions $(\hat x,\hat y,\hat q)$ and readings $m$, the relative residual is $\lVert\mathcal F(\hat s,\hat q)-m\rVert_2/\lVert m\rVert_2$ per row (float64). Report its mean, and the mean for the **true** validation sources (the noise floor, about `0.028`).

**Evaluation protocol.** Fit only on rows from `capstone_data.train_rows("inverse")` (and any other rows this statement names), make every choice on `capstone_data.val_rows("inverse")`, and reach the locked test rows only through `capstone_data.final_test_score(predict_fn, n_boot=1000, alpha=0.05, task="inverse")`, called exactly once, after all fitting and selection. Assert `capstone_data.test_call_count("inverse") == 0` immediately before that call and `== 1` after it.
`predict_fn` returns `model(magnitudes)` (float, `(N, 3)`) in eval mode under `torch.no_grad()`.

**Certify.** The last loss `<= 0.01 *` the first; the seam set is `{"train"}`; the mean validation residual is `<= 0.06`; the test score is `< capstone_data.baseline_score("inverse").score` and `<= 0.03`; the test-call protocol.
These bounds come from the frozen-seed reference run (test error about `0.010`, residual about `0.031`), with at least a two-fold margin.
Finally, explain in at most four sentences why the residual can be checked without true sources, and why it does not by itself prove the position is right.

**Required answer-check assertions.** Every certificate item, `len(losses) == 1500`, and the prediction shape `(200, 3)` on validation.

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261022
LOADER_CANDIDATES = (
    Path("../data/capstone_data.py"),
    Path("units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
    Path("book2/units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("capstone_data", loader_path)
capstone_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(capstone_data)
torch.set_num_threads(1)

CPU_CONFIG = {"device": "cpu", "amp_dtype": None, "simulated_pairs": 0, "hidden": 64,
              "steps": 1500, "batch_size": None, "lr": 3e-3}
RESIDUAL_BOUND = 0.06
ERROR_BOUND = 0.03

In [ ]:
# Write the required implementation, probes, certificates, and answer-check assertions here.

## Accelerator extension

*Optional; Colab L4 only; never executed by CI and never required for credit.*

On a Colab L4, scale the same `LearnedInverse` / `train_inverse` code by passing `L4_CONFIG` in place of `CPU_CONFIG`. It has exactly the keys of `CPU_CONFIG`, so only the dictionary changes: `simulated_pairs` adds 200,000 training pairs from the known forward process (`capstone_data.simulate("inverse", 200000, seed=3)`), `hidden` widens the network, and `batch_size` switches on mini-batches drawn from the global generator. Keep the seam rule "no `val` or `test` rows" (simulated rows hash to `None`).

```python
L4_CONFIG = {"device": "cuda", "amp_dtype": None, "simulated_pairs": 200000, "hidden": 256,
             "steps": 20000, "batch_size": 4096, "lr": 1e-3}
```

**What to observe.** Validation position error and mean residual as the number of simulated pairs grows (does the residual approach the noise floor?); whether `bfloat16` autocast changes the error of this small regression network (try it, then decide); and the error near the domain edges.

## Your response

Show every requested derivation, implementation, probe, certificate, or audit. Use only the permitted data, keep the training, validation, and locked-test roles separate, and end coding work with an answer-check section of executable assertions.